# MIE 446 computational companion

Course adaptation dated 23 September 2026. Original author: Ivan C. Christov, Purdue University. Upstream revision: [0f93a34750531e42e8806e5a11c46ac56769e669](https://github.com/ichristov/intermediate-fluid-mechanics/tree/0f93a34750531e42e8806e5a11c46ac56769e669). **GPL-3.0; see LICENSE.** Original author credit and caveats are retained.

Changes: cleared old outputs, resolved relative reading links, updated the Colab link. Scientific code and parameters are unchanged. run_notebook.py executes the scientific cells headlessly, omitting unused widget imports and IPython display setup. Generated plots, GIF and metrics are in results/. This is a prescribed-field visualization, not a Navier–Stokes momentum solver.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ehsan-Roohi/Aerospace-Structures/blob/main/computational/navier-stokes/NS_blowup_vortex.ipynb)

# LEARNING OBJECTIVE

After exploring this notebook, you will be able to describe a _self-similar_ swirling vortex whose core collapses to a point and whose velocity becomes unbounded in finite time, while the kinetic energy in that core vanishes, by combining ideas you have already seen: similarity variables (as in the [decay of an ideal vortex](https://github.com/ichristov/intermediate-fluid-mechanics/blob/0f93a34750531e42e8806e5a11c46ac56769e669/decay_ideal_vortex.ipynb)) and the continuity equation in cylindrical coordinates (and two concepts we have only touched on tangentially: conservation of angular momentum and [pathlines](https://github.com/ichristov/intermediate-fluid-mechanics/blob/0f93a34750531e42e8806e5a11c46ac56769e669/extras/flow_visualization.ipynb)).

> **Note: this is a rapidly developing topic and this notebook might change.** 

# PRELIMINARIES

[run the next cell to setup Python environment customizations and load packages]

In [ ]:
# interactive plots setup
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

# sympy setup
import sympy as sp
sp.init_printing()
from sympy.vector import *

# plotting customizations
from matplotlib import colormaps, animation, rc
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
size=16
params = {'legend.fontsize': 'large',
#          'figure.figsize': (20,8),
          'axes.labelsize': size,
          'axes.titlesize': size,
          'xtick.labelsize': size*0.875,
          'ytick.labelsize': size*0.875,
          'axes.titlepad': 25}
plt.rcParams.update(params)
%matplotlib inline

# numerics
import numpy as np
import scipy.integrate as spint

# for Colab only: to save plots as files and download them
#from google.colab import files

## Credit

Initial version written September 8, 2026 by [Ivan C. Christov](http://christov.tmnt-lab.org), Purdue University, in collaboration with Anthropic's [Claude](https://claude.ai) (Fable 5.1), which drafted the visualization code and discussion on the day the [OpenAI paper](https://openai.com/index/navier-stokes-solution/) was released, following the existing examples from the author in this GitHub repo.

Ivan C. Christov checked the notebook end-to-end and edited everything. He takes responsibility for any errors or misstatements. The underlying structure is from the OpenAI paper cited in the references at the end, which builds on the forcing approach of [C&oacute;rdoba & Mart&iacute;nez-Zoroa (2024)](https://arxiv.org/abs/2410.22920); a concurrent result for the forced Euler equations is due to Alp&ouml;ge and Buckmaster, announced in [Buckmaster (2026)](https://mastodon.social/@tristanbuckmaster/117233413705701198).

## Caveat

The velocity field constructed in this notebook is a _cartoon_: it has the same self-similar structure and scaling exponents as the leading-order flow in the [OpenAI paper](https://openai.com/index/navier-stokes-solution/), but their actual profile functions come from a 170-page construction (plus oscillatory corrections in an annulus around the core). The author makes no attempt whatsoever to build that here.

> **Nothing here is a solution of the Navier&ndash;Stokes equations.** 

This notebook only shows what the blow-up _might look like_, using concepts from ME 50900 &ndash; Intermediate Fluid Mechanics.

# INTRODUCTION

The two vortices we have encountered in this course either persist (the ideal vortex) or decay (the Lamb&ndash;Oseen vortex). Viscosity smooths velocity gradients, so a natural expectation is that a smooth flow stays smooth. Whether that is _always_ true for the 3D incompressible Navier&ndash;Stokes equations ([iNS](https://github.com/ichristov/intermediate-fluid-mechanics/blob/0f93a34750531e42e8806e5a11c46ac56769e669/handouts/iNS_summary_sheet.pdf)),
\begin{align*}
    \rho \frac{D \underline{v}}{D t} &= -\nabla p + \mu \nabla^2 \underline{v} + \rho \underline{F},\\
    \nabla\cdot\underline{v} &= 0,
\end{align*}
is the [Navier&ndash;Stokes regularity problem](https://www.claymath.org/millennium/navier-stokes-equation/), open since Leray's 1934 work. The alternative, a **finite-time singularity** or "blow-up," means the velocity becomes unbounded at some point at a finite time $t=t_c$, which would signal a breakdown of the continuum description (Panton, Ch. 1), not a physical event.

In September 2026, OpenAI announced an AI-generated construction of exactly such a blow-up, for a fluid started from rest and stirred by a smooth body force $\underline{F}$ ([announcement](https://openai.com/index/navier-stokes-solution/), [paper](https://cdn.openai.com/pdf/32d9f210-8b73-45e0-91bc-82a30aef8a9a/navier-stokes.pdf)). The proof is long and technical, and what it does and does not settle is discussed in the further exploration section at the end. What matters for us is that the blowing-up flow is a rather _quirky_ vortex, and that its structure can be understood with the tools of this course.

# A QUIRKY AXISYMMETRIC VORTEX WITH SWIRL

The leading-order flow in the paper (its Sec. 2.1) is axisymmetric (that is, nothing depends on $\theta$), in cylindrical coordinates $(r,\theta,z)$ about the vertical axis:
$$
    \underline{v} = v_r(r,z,t)\,\underline{e}_r + v_\theta(r,z,t)\,\underline{e}_\theta + v_z(r,z,t)\,\underline{e}_z .
$$
Unlike the ideal vortex, all three components are nonzero. Physically:

* $v_\theta$ is the **swirl**, which becomes unbounded;
* $v_r<0$ near the midplane $z=0$ is an **inflow** that carries angular momentum toward the axis (recall: without torque, $r v_\theta$ is conserved along a fluid particle's path, so moving inward _spins up_ the particle);
* $v_z$ is an **axial jet** away from the midplane, up for $z>0$ and down for $z<0$, which is required by incompressibility: the fluid arriving at the axis has to go somewhere.

The paper's [Fig. 1](https://openai.com/index/navier-stokes-solution/) is a schematic of the core at three successive times $t_1<t_2<t_3<t_c$: fluid spirals inward and is ejected along the axis, and the region of intense flow shrinks in radius faster than in height. We will build our own version of that picture below.

For an axisymmetric flow, the incompressibility constraint in cylindrical coordinates reads
$$
    \nabla\cdot\underline{v} = \frac{1}{r}\frac{\partial}{\partial r}\left(r v_r\right) + \frac{1}{r}\cancel{\frac{\partial v_\theta}{\partial \theta}} + \frac{\partial v_z}{\partial z} = 0 ,
$$
which we will use to _construct_ $v_r$ from a chosen $v_z$. (Note that $v_\theta$ drops out entirely: the swirl can be prescribed independently!)

# SELF-SIMILAR STRUCTURE

## Recall: the decaying vortex

In the [Lamb&ndash;Oseen vortex](https://github.com/ichristov/intermediate-fluid-mechanics/blob/0f93a34750531e42e8806e5a11c46ac56769e669/decay_ideal_vortex.ipynb), we found $v_\theta(r,t) = \frac{\Gamma}{2\pi r}f(\eta)$ with the similarity variable $\eta = r/\sqrt{\nu t}$. The viscous core _grows_ like $\sqrt{\nu t}$, and $v_\theta$ _decays_. At any instant of time, $v_\theta$ is found from one "universal" shape $f(\eta)$, suitably stretched.

## The blow-up vortex

The blow-up solution is also self-similar, but with time running in reverse. Let $t_c$ be the singularity time (instant of blow-up) and
$$
    \tau = t_c - t
$$
the time _remaining_ until blow-up. The paper introduces similarity variables
$$
    \xi = \frac{r^2}{2\tau}, \qquad \eta = \frac{z}{\tau^{D}},
$$
with
$$
    A = \tfrac{1}{2}+h,\quad D = \tfrac{1}{2}-h, \quad 0<h<\tfrac{1}{100},
$$
and writes the leading-order velocity as
\begin{align*}
    v_\theta &= \tau^{-A}\, E(\xi,\eta), \\
    v_z &= \tau^{-A}\, U(\xi,\eta), \\
    r\, v_r &= V(\xi,\eta),
\end{align*}
where $E,U,V$ are fixed profile functions. (The exponent $h$ is there only to break the symmetry between the radial and axial length scales.)

Reading off the scalings, the **core** of the vortex has
\begin{align*}
    \text{radius } \ell_r &\sim \tau^{1/2}, \\
    \text{height } \ell_z &\sim \tau^{1/2-h}, \\
    \text{speeds } |v_\theta|, |v_z| &\sim \tau^{-1/2-h}, \quad \textit{but} \quad |v_r|\sim \tau^{-1/2},\\
    \text{aspect ratio } \ell_z/\ell_r &\sim \tau^{-h} \to \infty .
\end{align*}
So both lengths shrink, but the core becomes an ever _slenderer_ column, "like spaghetti," while the speed inside diverges. 

On September 18, 2026, [Numberphile](https://www.youtube.com/watch?v=3geDF-DAwpg) released a video with a nice explanation of what these scalings mean.

Be warned that $\tau^{-h}$ is a very weak divergence: with $h=1/20$ the aspect ratio grows by only 16% over the factor of 20 in $\tau$ plotted below, and we cannot raise $h$ to help, since the energy estimate just below needs $h<1/6$. The paper's schematic exaggerates the slenderness for this reason, and the plots below report the _trend_ in $\ell_z/\ell_r$ rather than its absolute value.

The kinetic energy in the core is roughly (speed)$^2\times$(volume):
$$
    E_{\mathrm{core}} \sim \tau^{-1-2h}\cdot \tau^{3/2-h} = \tau^{1/2-3h} \to 0 .
$$
This is how an infinite speed can be consistent with finite (in fact, vanishing) kinetic energy: the fast region has vanishing volume.

Compare with the Lamb&ndash;Oseen scalings: there, $r\sim\sqrt{\nu t}$ and $v_\theta \sim 1/\sqrt{\nu t}$, that is, the same $\pm 1/2$ exponents with $t\mapsto \tau$ and the direction reversed. In both cases, the radial diffusion time $\ell_r^2/\nu$ is comparable to the time elapsed/remaining, so _viscosity is in the leading-order balance_; it is not negligible in the blow-up core. What diverges is the _angular_ Reynolds number $|v_\theta|\ell_r/\nu \sim \tau^{-h}$: the fluid makes ever more turns per radial diffusion time.

## Relation to the Burgers vortex

There is a closer relative in Panton, Sec. 11.10: the **Burgers vortex**, a steady exact solution of the Navier&ndash;Stokes equations,
$$
    v_r = -\frac{\gamma}{2}\, r, \qquad v_z = \gamma z, \qquad v_\theta = \frac{\Gamma}{2\pi r}\left[1 - e^{-\gamma r^2/4\nu}\right],
$$
in which a _constant_ axisymmetric strain rate $\gamma$ drives fluid inward and stretches it along the axis, concentrating the swirl, while viscosity diffuses the swirl outward. The two effects balance at a fixed core radius $\sqrt{4\nu/\gamma}$. 

(Note that the swirl profile is exactly Lamb&ndash;Oseen's with $t\mapsto 1/\gamma$.) The blow-up vortex has the same three ingredients, inflow, axial stretching, and viscous diffusion of swirl, but the strain rate is not constant: from the scalings above, $\gamma \sim v_r/r \sim 1/\tau$. With the strain rate growing like $1/\tau$, the Burgers balance $\ell_r^2 \sim \nu/\gamma \sim \nu\tau$ still holds at every instant, but the core radius it selects shrinks to zero at $t = t_c$. In this sense, the blow-up flow is a Burgers vortex whose strain rate runs away.

# A CARTOON MODEL OF THIS QUIRKY VORTEX

We now pick simple profiles that have the right structure. We work in dimensionless variables with $t_c=1$ and $\nu=1$; the paper's rescaling $\underline{v}_\nu(\underline{x},t) = \sqrt{\nu}\,\underline{v}(\underline{x}/\sqrt{\nu}, t)$ recovers any other $\nu$ with the same $t_c$, so $\nu$ sets the overall scale but not the exponents. In fact $\nu$ never appears below, since we _prescribe_ the profiles rather than solve for them.

* **Swirl:** $E(\xi) = \sqrt{2\xi}\,(1+\xi)^{-1-h}$. This is our one structural simplification: the paper's profile is $E(\xi,\eta)$, and dropping the $\eta$ dependence makes our swirl a column of the same strength at every height. The factor $\sqrt{2\xi} = r/\sqrt{\tau}$ makes $v_\theta$ vanish linearly on the axis (regularity, like the Lamb&ndash;Oseen $f\sim\eta^2$), and $E\sim \xi^{-1/2-h}$ for large $\xi$ matches the paper's exterior swirl, which is the exact heat-equation solution $v_\theta\sim r^{-1-2h}$ (Eq. (4.29) in the paper). Note: at fixed $r$, $v_\theta \sim \tau^{-A}(r^2/2\tau)^{-A}\propto r^{-2A}$ is _independent of $\tau$_, so the exterior is frozen while the core collapses.
* **Axial jet:** $U(\xi,\eta) = (4\eta + j_0)\, e^{-\eta^2} e^{-\xi/\xi_c}$. The paper's axis data is $U\approx 4\eta$: up above the midplane, down below. The small offset $j_0>0$ is the paper's "slight upward bias" (they need $v_z\neq 0$ at $z=0$ for technical reasons); $\xi_c$ sets the core width.
* **Radial inflow:** $v_r$ from incompressibility. We let SymPy do this integral for us.

In [ ]:
# similarity exponents; h = 1/20 is already exaggerated relative to the paper's 0 < h < 1/100,
# but the tau^{-h} drift of the core aspect ratio is still far too slow to see by eye. Raising h
# further is not an option: the core-energy estimate needs h < 1/6
h = sp.Rational(1, 20)
A = sp.Rational(1, 2) + h
D = sp.Rational(1, 2) - h

# cartoon parameters
j0 = sp.Rational(3, 10)   # upward bias of the axial jet
xi_c = 1                  # core width in similarity units
t_c = 1                   # blow-up time

# symbols: we work with tau = t_c - t > 0, which lets SymPy simplify without case splits
r, z, tau = sp.symbols('r z tau', positive=True)
xi = r**2/(2*tau)
eta = z/tau**D

# profiles
E = sp.sqrt(2*xi)*(1 + xi)**(-1 - h)
U = (4*eta + j0)*sp.exp(-eta**2)*sp.exp(-xi/xi_c)

# velocity components
v_theta = tau**(-A)*E
v_z = tau**(-A)*U
display(v_theta, v_z)

Integrating the continuity equation from the axis (where $r v_r = 0$ to cancel the coordinate singularity, which is not a "real" singularity) gives
$$
    r v_r = -\int_0^r r'\,\frac{\partial v_z}{\partial z}\,dr' .
$$

In [ ]:
rp = sp.symbols("r'", positive=True)
r_vr = -sp.integrate(rp*sp.diff(v_z, z).subs(r, rp), (rp, 0, r))
v_r = sp.simplify(r_vr/r)
display(v_r)

SymPy leaves an $e^{+\xi}$ inside the bracket, multiplying the $e^{-\xi}$ out front. They cancel on paper, but in floating point $e^{+\xi}$ overflows once $\xi = r^2/(2\tau)$ passes about $709$ (at $r=1$, that is $\tau\lesssim7\times10^{-4}$), and `inf`$\,\times\,$`0` returns `NaN`. Expanding distributes the exponential term by term: the same expression, but safe at the $\tau=10^{-6}$ we reach below.

In [ ]:
v_r = sp.expand(v_r)

# always check that a rewritten expression is still the one you derived
print('expanded form agrees with the integral:', sp.simplify(v_r - r_vr/r) == 0)

Note that $r v_r$ came out as a function of $\xi$ and $\eta$ only, with no leftover power of $\tau$, exactly as in the ansatz $r v_r = V(\xi,\eta)$ (this is because $A + D = 1$). Let us verify incompressibility using SymPy's [`vector`](https://docs.sympy.org/latest/modules/vector/index.html) module, as we did for the ideal vortex:

In [ ]:
# create a cylindrical coordinate system for us to define vectors in
c = CoordSys3D('c')
cp = c.create_new('cp', transformation='cylindrical',
                  vector_names=("r", "th", "z"),
                  variable_names=("R", "Th", "Z"))

v = (v_r.subs({r: cp.R, z: cp.Z})*cp.r
     + v_theta.subs({r: cp.R, z: cp.Z})*cp.th
     + v_z.subs({r: cp.R, z: cp.Z})*cp.z)

sp.simplify(divergence(v))

Good, the cartoon flow is incompressible. Is it irrotational? Certainly not, though only two of the three vorticity components survive here: $\omega_\theta = \partial v_r/\partial z - \partial v_z/\partial r$ and $\omega_z = \frac{1}{r}\frac{\partial}{\partial r}(r v_\theta)$. The radial one vanishes identically, $\omega_r = -\partial v_\theta/\partial z = 0$, because we chose a swirl $E(\xi)$ with no $\eta$ dependence; the paper's $E(\xi,\eta)$ has one, and there all three are nonzero. (Try `curl(v)` if you are curious, but be warned that the expression is long.)

For numerics, as usual, we `lambdify` the symbolic expressions into fast NumPy functions:

In [ ]:
# lambdify in (r, z, tau): working in tau directly avoids the cancellation in t_c - (t_c - tau)
# once tau gets very small
v_r_tau = sp.lambdify((r, z, tau), v_r, 'numpy')
v_theta_tau = sp.lambdify((r, z, tau), v_theta, 'numpy')
v_z_tau = sp.lambdify((r, z, tau), v_z, 'numpy')

# wrappers in physical time t, used for the pathline integration further below
def v_r_num(r, z, t): return v_r_tau(r, z, t_c - t)
def v_theta_num(r, z, t): return v_theta_tau(r, z, t_c - t)
def v_z_num(r, z, t): return v_z_tau(r, z, t_c - t)

# core length scales as functions of tau (paper: xi <= xi_c, |eta| <= 1)
hf, Af, Df, xi_cf = float(h), float(A), float(D), float(xi_c)
def ell_r(tau_): return np.sqrt(2*xi_cf*tau_)
def ell_z(tau_): return tau_**Df

# VISUALIZATION

## Swirl profile on the midplane

As for the Lamb&ndash;Oseen vortex, we plot $v_\theta(r, z=0, t)$ at successive times in physical variables (left) and in similarity variables (right). Here the _time arrow is reversed_: the curves sharpen and grow instead of spreading and decaying, but again they all collapse onto a single profile $E(\xi)$ when rescaled.

In [ ]:
# define horizontal extent of the r plot and grid (avoiding r=0)
rmax = 1.0
r_ = np.linspace(1e-6, rmax, 500)

# define extent of the similarity plot
xi_max = 6.0

# sequential colormap: dark = far from blow-up, bright = close to blow-up
cmap = colormaps['viridis']

# set up the figure and axis
fig, ax = plt.subplots(1, 2, tight_layout=True)
ax[0].set_xlabel('$r$')
ax[0].set_xlim(0, rmax)
ax[0].set_ylabel(r'$v_\theta(r, z=0, t)$')
ax[0].set_ylim(0, 14)
ax[0].set_aspect(1.0/ax[0].get_data_ratio(), adjustable='box')

ax[1].set_xlabel(r'$\xi = r^2/(2\tau)$')
ax[1].set_xlim(0, xi_max)
ax[1].set_ylabel(r'$\tau^{A} v_\theta = E(\xi)$')
ax[1].set_ylim(0, 1)
ax[1].set_aspect(1.0/ax[1].get_data_ratio(), adjustable='box')
ax[1].yaxis.set_label_position('right')

# geometric spacing in tau so each curve is the same "zoom factor" from the last
tau_list = np.geomspace(0.3, 0.005, 12)
for tau_ in tau_list:
    vth = v_theta_tau(r_, 0*r_, tau_)
    tcol = cmap(np.log(tau_/tau_list[0])/np.log(tau_list[-1]/tau_list[0]))
    ax[0].plot(r_, vth, linewidth=1, color=tcol)
    ax[1].plot(r_**2/(2*tau_), tau_**Af*vth, linewidth=1, color=tcol)

# frozen exterior: v_theta -> sqrt(2) 2^A r^{-2A}, independent of time
ax[0].plot(r_, np.sqrt(2)*2**Af*r_**(-2*Af), color='gray', linewidth=1, linestyle='dashed',
           label=r'$\sqrt{2}\,2^{A} r^{-1-2h}$')
ax[0].legend(loc='upper right')

ax[0].grid(alpha=0.5, linestyle='dotted')
ax[1].grid(alpha=0.5, linestyle='dotted')

ax[0].annotate(r'$t \to t_c$', xy=(0.45, 6), xytext=(0.45, 0.125),
               arrowprops=dict(arrowstyle="->"));

Each curve peels away from the dashed line at its own core radius $\ell_r=\sqrt{2\tau}$ and rejoins it at about $r = 3\ell_r$, where the swirl is within 10% of $\sqrt{2}\,2^{A}r^{-1-2h}$ and no longer depends on time. This is the "heat exterior" of the paper: at fixed $r$ the two powers of $\tau$ in $v_\theta = \tau^{-A}E(\xi)$ cancel. So it is the curves closest to blow-up that agree with the asymptote, the last of them ($\tau=0.005$, $\ell_r\approx0.1$) by $r\approx0.3$; the earliest ($\tau=0.3$, $\ell_r\approx0.77$) is all core across the whole window.

Rescaled by $\tau^{A}$ and plotted against $\xi$ (right), all twelve fall on one curve: the core sharpens, but its _shape_ is invariant.

## Meridional flow: inflow and axial jet

Next we look at the $(r,z)$ "meridional" plane, showing the swirl $v_\theta$ as a color map and the in-plane velocity $(v_r, v_z)$ as arrows. We mirror the plot about the axis so that it looks like a cross-section through it, so $r$ increases both to the left and to the right. Because the speeds diverge, we use a log color scale and _unit_ arrows (direction only).

In [ ]:
# meridional grid; even num avoids r=0 exactly
rmax = 1.0
zmax = 1.0
rr, zz = np.meshgrid(np.linspace(-rmax, rmax, num=200),
                     np.linspace(-zmax, zmax, num=200))

# helper to evaluate the field on the mirrored grid
def meridional_field(tau_):
    vr = v_r_tau(np.abs(rr), zz, tau_)*np.sign(rr)   # v_r flips sign across the axis in this plot
    vth = v_theta_tau(np.abs(rr), zz, tau_)
    vz = v_z_tau(np.abs(rr), zz, tau_)
    return vr, vth, vz

fig, ax = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

# r runs to the left and to the right of the axis, so label both sides with |r|
rticks = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])

# ell_z/ell_r depends on where we draw the edge of the core; the trend (tau/tau_0)^(-h) does not
tau_panels = [0.2, 0.05, 0.01]
aspect0 = ell_z(tau_panels[0])/ell_r(tau_panels[0])

for k, tau_ in enumerate(tau_panels):
    vr, vth, vz = meridional_field(tau_)
    pcm = ax[k].pcolormesh(rr, zz, vth, cmap='magma', norm=LogNorm(vmin=0.3, vmax=30),
                           shading='auto')
    # unit arrows, skipping points to reduce clutter
    sk = (slice(6, None, 14), slice(6, None, 14))
    mag = np.hypot(vr[sk], vz[sk])
    ax[k].quiver(rr[sk], zz[sk], vr[sk]/mag, vz[sk]/mag, color='white', scale=25, width=0.005)
    # core box
    lr, lz = ell_r(tau_), ell_z(tau_)
    ax[k].plot([-lr, lr, lr, -lr, -lr], [-lz, -lz, lz, lz, -lz], 'c--', linewidth=1.5)
    ax[k].set_title(rf'$\tau = {tau_}$, core aspect $\times{(lz/lr)/aspect0:.2f}$')
    ax[k].set_xlabel('$r$')
    ax[k].set_xticks(rticks, [f'{abs(v):.2f}' for v in rticks])   # r is positive on both sides
    ax[k].set_aspect('equal')

ax[0].set_ylabel('$z$')
fig.colorbar(pcm, ax=ax.tolist(), label=r'$v_\theta$', shrink=0.8);

Notice the three features of the paper's Fig. 1: fluid streams inward along the midplane, is turned up and down along the axis, and returns as outflow far above and below the core, because the flow is incompressible. The dashed box is the core $\{\xi\le \xi_c,\ |\eta|\le 1\}$, shrinking toward the origin.

The titles report how much more slender the core is than in the left panel: 16% across the factor of 20 in $\tau$ these panels span, which is why the three boxes look alike. $\tau^{-h}$ is unbounded, but at $h=1/20$ it needs six decades of $\tau$ per doubling.

## Animation

We can also illustrate the collapse as an animation, and now push much closer to the singular time, down to $\tau = 10^{-6}$. Since the core shrinks like $\sqrt{\tau}$, we advance $\tau$ geometrically between frames, so every frame is the same zoom factor from the last.

The two panels answer different questions, so they are scaled differently:

* **Left**, a _fixed_ physical window showing the raw $v_\theta$. The core collapses to a point and the surroundings stop changing.
* **Right**, a window that _follows_ the core, showing the rescaled swirl $\tau^{A}v_\theta = E(\xi)$ and the in-plane direction measured in core units, $(v_r/\ell_r,\, v_z/\ell_z)$. Both are exactly $\tau$-independent, so this panel should not change at all.

We start at $\tau=0.05$ rather than $0.3$, where the core would nearly fill the frame and leave no exterior in view to stay frozen.

In [ ]:
# set up the figure and axes
fig, ax = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)

# left panel: fixed physical window
ax[0].set_xlabel('$r$')
ax[0].set_xticks(rticks, [f'{abs(v):.2f}' for v in rticks])   # r is positive on both sides
ax[0].set_ylabel('$z$')
ax[0].set_aspect('equal')
ax[0].set_title('fixed window')

# right panel: window that follows the core
ax[1].set_xlabel(r'$r/\ell_r$')
sticks = np.array([-4.0, -2.0, 0.0, 2.0, 4.0])
ax[1].set_xticks(sticks, [f'{abs(v):g}' for v in sticks])
ax[1].set_ylabel(r'$z/\ell_z$')
ax[1].set_xlim(-4, 4)
ax[1].set_ylim(-4, 4)
ax[1].set_aspect('equal')
ax[1].set_title('window following the core')

# similarity grid for the right panel, in units of the core (even num avoids the axis)
ss, ee = np.meshgrid(np.linspace(-4, 4, num=120), np.linspace(-4, 4, num=120))

# tau runs geometrically from tau_start down to tau_end
tau_start = 0.05
tau_end = 1e-6

# as above, the core aspect ratio is reported relative to its first-frame value
aspect_start = ell_z(tau_start)/ell_r(tau_start)

# initialize plot objects with the first frame (replaced in animation)
vr, vth, vz = meridional_field(tau_start)
pcm0 = ax[0].pcolormesh(rr, zz, vth, cmap='magma', norm=LogNorm(vmin=0.5, vmax=200), shading='auto')
core0, = ax[0].plot([], [], 'c--', linewidth=1.5)
fig.colorbar(pcm0, ax=ax[0], label=r'$v_\theta$', shrink=0.8)

# the rescaled swirl is O(1) and time-independent, so it gets a plain linear scale of its own
pcm1 = ax[1].pcolormesh(ss, ee, np.ones_like(ss), cmap='magma', vmin=0, vmax=0.7, shading='auto')
sk1 = (slice(4, None, 8), slice(4, None, 8))
quiv1 = ax[1].quiver(ss[sk1], ee[sk1], 0*ss[sk1], 0*ss[sk1], color='white', scale=22, width=0.006)
ax[1].plot([-1, 1, 1, -1, -1], [-1, -1, 1, 1, -1], 'c--', linewidth=1.5)
fig.colorbar(pcm1, ax=ax[1], label=r'$\tau^{A} v_\theta = E(\xi)$', shrink=0.8)

time_label = ax[0].annotate('', xy=(-0.94, 0.84), fontsize=11,
                            bbox=dict(boxstyle="round", facecolor="white"))
plt.close()

# number of animation frames
numframes = 50

# animation function called sequentially by `FuncAnimation' below
def animate(k):
    # the k passed is the frame counter, not time;
    # we space tau geometrically so each frame zooms in by the same factor
    tau_ = tau_start*(tau_end/tau_start)**(k/(numframes - 1))
    lr, lz = ell_r(tau_), ell_z(tau_)

    # left: fixed window, raw swirl. No arrows: in the later frames the meridional flow is
    # confined to far less than one grid cell
    vr, vth, vz = meridional_field(tau_)
    pcm0.set_array(vth.ravel())
    core0.set_data([-lr, lr, lr, -lr, -lr], [-lz, -lz, lz, lz, -lz])

    # right: co-moving window (physical coords = core scales * similarity coords)
    R1, Z1 = np.abs(lr*ss), lz*ee
    vr1 = v_r_tau(R1, Z1, tau_)*np.sign(ss)
    vth1 = v_theta_tau(R1, Z1, tau_)
    vz1 = v_z_tau(R1, Z1, tau_)
    pcm1.set_array((tau_**Af*vth1).ravel())
    # in-plane direction measured in core units, which is the self-similar quantity
    u1, w1 = vr1[sk1]/lr, vz1[sk1]/lz
    mag1 = np.hypot(u1, w1)
    quiv1.set_UVC(u1/mag1, w1/mag1)

    # dynamic label in the corner
    time_label.set_text(rf'$\tau = {tau_:.1e}$,  $\max\, v_\theta \approx {vth1.max():6.0f}$,'
                        rf'  core aspect $\times{(lz/lr)/aspect_start:.2f}$')

    return (pcm0, core0, pcm1, quiv1, time_label)


anim = animation.FuncAnimation(fig, animate,
                               frames=numframes, interval=150, blit=False)

In [ ]:
# this is necessary to get the animation to work on Google's Colab
rc('animation', html='jshtml')
display(anim)

In [ ]:
# To save the animation, use something like
# anim.save(f"NS_blowup_meridional.mp4")

# or
# writer = animation.FFMpegWriter(fps=10, metadata=dict(artist='Me'), bitrate=1800)
# anim.save(f"NS_blowup_meridional.mp4", writer=writer)

Left: the core collapses to a point, and after the first few frames the surroundings stop changing at all. That frozen "heat exterior" is why the background locks in place while the thread at the axis brightens. (The thread is the core, no longer resolved once $\ell_r=\sqrt{2\tau}$ drops below the cell size $0.01$, that is, for $\tau\lesssim5\times10^{-5}$.)

Right: nothing changes, to machine precision. The rescaled swirl $\tau^{A}v_\theta$ and the in-plane direction in core units are identical in every frame. That is what self-similarity means.

Only the label moves. The peak swirl climbs from about 4 to about 1400, cleanly unbounded, while the core aspect ratio gains a factor of just 1.7 over nearly five decades of $\tau$. That is the honest version of "like spaghetti."

## Pathlines

Finally, let us follow some fluid particles, as we did in the [flow visualization](https://github.com/ichristov/intermediate-fluid-mechanics/blob/0f93a34750531e42e8806e5a11c46ac56769e669/extras/flow_visualization.ipynb) notebook, to see the "spiral inward, stretch along the axis" motion. We integrate
$$
    \frac{d\underline{x}}{dt} = \underline{v}(\underline{x},t)
$$
in Cartesian coordinates using [`solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html). Because the velocity diverges as $t\to t_c$, we stop the integration slightly before the singularity time. (The default adaptive Runge&ndash;Kutta time integration will take shorter and shorter steps as the particles speed up.)

In [ ]:
def pathline_ode(t, position):
    x, y, z = position
    r = np.sqrt(x**2 + y**2)
    th = np.arctan2(y, x)

    # polar velocity components
    vr = v_r_num(r, z, t)
    vth = v_theta_num(r, z, t)
    vz = v_z_num(r, z, t)

    # transform polar velocities to Cartesian ones
    u = vr*np.cos(th) - vth*np.sin(th)
    v = vr*np.sin(th) + vth*np.cos(th)
    return [u, v, vz]

In [ ]:
# release particles on a ring near the midplane
t0 = t_c - 0.3
t_end = t_c - 0.002
num_particles = 8
rng = np.random.default_rng(1)
th0 = np.linspace(0, 2*np.pi, num_particles, endpoint=False)
r0 = 0.45*np.ones(num_particles)
z0 = rng.uniform(-0.1, 0.1, num_particles)

t_eval = t_c - np.geomspace(t_c - t0, t_c - t_end, 300)   # dense near the blow-up
pathlines = []
for i in range(num_particles):
    sol = spint.solve_ivp(pathline_ode, [t0, t_end],
                          [r0[i]*np.cos(th0[i]), r0[i]*np.sin(th0[i]), z0[i]],
                          t_eval=t_eval, rtol=1e-6, atol=1e-9)
    pathlines.append(sol.y)

In [ ]:
fig = plt.figure(figsize=(12, 5.5), tight_layout=True)
ax3 = fig.add_subplot(1, 2, 1, projection='3d')
ax2 = fig.add_subplot(1, 2, 2)

for i, p in enumerate(pathlines):
    ax3.plot(p[0], p[1], p[2], linewidth=1, color=cmap(i/num_particles))
    ax3.plot(p[0][0], p[1][0], p[2][0], 'bo', markersize=3)
    ax3.plot(p[0][-1], p[1][-1], p[2][-1], 'r^', markersize=4)
    # projection onto the (r, z) meridional plane
    ax2.plot(np.hypot(p[0], p[1]), p[2], linewidth=1, color=cmap(i/num_particles))
    ax2.plot(np.hypot(p[0][0], p[1][0]), p[2][0], 'bo', markersize=3)
    ax2.plot(np.hypot(p[0][-1], p[1][-1]), p[2][-1], 'r^', markersize=4)

ax3.set_xlabel('$x$'); ax3.set_ylabel('$y$'); ax3.set_zlabel('$z$')
ax3.set_xlim(-0.65, 0.65); ax3.set_ylim(-0.65, 0.65); ax3.set_zlim(-0.65, 0.65)
ax3.set_title('pathlines in 3D')

ax2.set_xlabel('$r$'); ax2.set_ylabel('$z$')
ax2.set_xlim(0, 0.65); ax2.set_ylim(-0.65, 0.65)
ax2.set_title(r'projected onto the $(r,z)$ plane')
ax2.set_aspect(1.0, adjustable='box')
ax2.plot([], [], 'bo', label='start point'); ax2.plot([], [], 'r^', label='end point')
ax2.legend(loc='lower left')
ax2.grid(alpha=0.5, linestyle='dotted');

Particles spiral inward, their swirl speed increasing as $r$ decreases by conservation of angular momentum, and are then turned and ejected along the axis. None of them actually reaches $r=0$: the upward bias $j_0$ lifts a particle off the midplane, its $|\eta|$ then grows as $\tau^{D}$ shrinks, and the axial jet fires it out. How far in a particle gets before that happens depends on where it was released, and not monotonically: the deepest here penetrates to $r\approx0.03$, about half a core radius, while others turn around near $r\approx0.3$. 

This picture is reminiscent of the "bathtub drain" of the NCFMF [Vorticity](https://video.odl.mit.edu/videos/2e75fb6283de4b54a09ee0da1c8e78e9/) film, except that here the drain is not a hole in the bottom of the tank but the axis itself, and there is no bottom: the flow is symmetric (up to the small bias $j_0$) about $z=0$.

## Checking the scalings

Let us verify the two properties that make this a candidate blow-up: the speed becomes unbounded, but the kinetic energy stays bounded. We compute the maximum swirl and the kinetic energy 
$$
    E_{\mathrm{core}} = \frac{1}{2}\int_{\mathrm{core}}|\underline{v}|^2\,dV = \pi\int\!\!\int_{\mathrm{core}} |\underline{v}|^2 \, r dr dz
$$
in the core box as functions of $\tau$.

In [ ]:
tau_list = np.geomspace(0.3, 1e-4, 30)
vmax_list = []
Ecore_list = []
for tau_ in tau_list:
    lr, lz = ell_r(tau_), ell_z(tau_)
    # grid that resolves the (shrinking) core
    r1 = np.linspace(1e-9, lr, 300)
    z1 = np.linspace(-lz, lz, 301)
    R1, Z1 = np.meshgrid(r1, z1)
    speed2 = v_r_tau(R1, Z1, tau_)**2 + v_theta_tau(R1, Z1, tau_)**2 + v_z_tau(R1, Z1, tau_)**2
    vmax_list.append(v_theta_tau(R1, Z1, tau_).max())
    Ecore_list.append(np.pi*np.trapezoid(np.trapezoid(R1*speed2, r1, axis=1), z1))

fig, ax = plt.subplots(1, 2, tight_layout=True)

ax[0].loglog(tau_list, vmax_list, 'o', markersize=4, label='cartoon')
ax[0].loglog(tau_list, vmax_list[0]*(tau_list/tau_list[0])**(-Af), 'k--', linewidth=1,
             label=r'$\tau^{-(1/2+h)}$')
ax[0].set_xlabel(r'$\tau = t_c - t$')
ax[0].set_ylabel(r'$\max \, v_\theta$')

ax[1].loglog(tau_list, Ecore_list, 'o', markersize=4, label='cartoon')
ax[1].loglog(tau_list, Ecore_list[0]*(tau_list/tau_list[0])**(0.5 - 3*hf), 'k--', linewidth=1,
             label=r'$\tau^{1/2-3h}$')
ax[1].set_xlabel(r'$\tau = t_c - t$')
ax[1].set_ylabel(r'$E_{\mathrm{core}}$')
ax[1].yaxis.set_label_position('right')

for a in ax:
    a.invert_xaxis()   # time runs left to right toward the blow-up
    a.grid(alpha=0.5, linestyle='dotted', which='both')
    a.legend()

As $\tau\to0$ (left to right), the maximum swirl diverges like $\tau^{-1/2-h}$ while the kinetic energy in the core vanishes like $\tau^{1/2-3h}$. Since $h<1/6$, the core energy goes to zero even though the speed goes to infinity. Do not read more into that than it says. Our cartoon cannot show the energy of the whole flow staying bounded: with no $\eta$ dependence in the swirl, $v_\theta$ is the same at every height, so its energy over all $z$ is infinite, and even per unit height it creeps up like $\tau^{-2h}$. The paper's construction is localized with smooth cutoffs far from the core (its Sec. 3.5), and there the total energy really is uniformly bounded (its Theorem 1.1).

The data fall slightly _below_ the $\tau^{1/2-3h}$ line because that exponent covers only the swirl and the jet. The radial inflow, $v_r\sim\tau^{-1/2}$, contributes the steeper $\tau^{1/2-h}$: it is a third of the total at the largest $\tau$, where the line is anchored, and then decays away.

**On your own:** split `speed2` into its three terms and plot them separately. The axial jet carries the most energy throughout; the jet and the swirl both follow $\tau^{1/2-3h}$; and the radial inflow follows the steeper $\tau^{1/2-h}$, starting out ahead of the swirl and being overtaken by it near $\tau\approx7\times10^{-4}$. Can you recover that crossover from the two exponents and their prefactors?

# FURTHER EXPLORATION

On your own:

1. **Burgers vortex.** Read Panton, Sec. 11.10 (and Sec. 11.12 for the von K&aacute;rm&aacute;n viscous pump). Which part of the cartoon plays the role of the strain rate $\gamma$? Evaluate $\gamma_{\mathrm{eff}} = -2v_r/r$ near the axis on the midplane and confirm it grows like $1/\tau$. Plot the Burgers swirl against our $E(\xi)$: same shape, different exterior.
2. **Vary $h$.** Set `h = sp.Rational(1, 200)`, closer to the paper's range, and re-run. What does the core aspect ratio do now? What if $h = 0$? What breaks in the energy estimate if $h \ge 1/6$?
3. **Turn off the bias.** Set `j0 = 0`. What symmetry does the flow gain about $z=0$, and where is $v_z = 0$? The paper's Sec. 2.1 explains why that symmetry has to be broken.
4. **Vorticity.** Compute $\underline{\omega} = \nabla\times\underline{v}$ with `curl(v)` and plot $\omega_z$ on the midplane in similarity variables. How does its peak scale with $\tau$? Compare with the Lamb&ndash;Oseen $\omega_z\sim 1/(\nu t)$.
5. **Is it a solution?** Everything above is kinematics: we prescribed profiles and checked continuity, never momentum. Take the $\theta$-component of the iNS equations in cylindrical coordinates (tabulated in Panton's Appendices; with $\nu=\rho=1$, and pressure drops out by axisymmetry), substitute the cartoon's velocity in SymPy, and find the body force $F_\theta$ that would sustain it. Does $\tau^{A+1}F_\theta$ collapse against $\xi$? Where is it largest, in the core or outside it? The paper cancels a residual like this with small oscillatory "pulses" whose averaged momentum fluxes $\langle v_r' v_\theta' \rangle$, $\langle v_r' v_z' \rangle$ (the same Reynolds stresses we meet in the final lecture of ME 50900!) carry the missing momentum: that is where most of its 170 pages go, and why the theorem needs a force at all.
6. **Read further.** Sec. 2 of the paper, "Physical description of the blowup", is four pages and readable with the background of this course; try to identify each term of the momentum equation in its discussion of the "leading balance". A companion OpenAI paper (and, independently, Alp&ouml;ge and Buckmaster) also proves blow-up for the forced _inviscid_ Euler equations; compare with the numerically discovered Hou&ndash;Luo scenario [(Luo & Hou, 2014)](https://doi.org/10.1073/pnas.1405238111) in a cylinder with counter-rotating halves. Compare their Fig. 1 with our meridional plots: what is the same, and what is different? The official problem statement [(Fefferman, 2000)](https://www.claymath.org/wp-content/uploads/2022/06/navierstokes.pdf) allows a smooth force in its alternatives (C) and (D), the version resolved here; whether the _unforced_ problem is the one fluid mechanicians really care about is argued over in _Quanta Magazine_ [(Kakaes, 2026)](https://www.quantamagazine.org/ai-has-solved-one-of-maths-1-million-millennium-prize-problems-20260908/) and _Scientific American_ [(Howlett, 2026)](https://www.scientificamerican.com/article/openai-claims-blockbuster-math-breakthrough-amid-swirl-of-controversy/).

# REFERENCES

T. Buckmaster, Announcement of joint work with L. Alp&ouml;ge on finite-time blow-up for the forced 3D Euler equations, Mastodon post, September 2026. [link](https://mastodon.social/@tristanbuckmaster/117233413705701198)

D. C&oacute;rdoba, L. Mart&iacute;nez-Zoroa, Finite time singularities of smooth solutions for the 2D incompressible porous media (IPM) equation with a smooth source, arXiv preprint, 2024. [arXiv:2410.22920](https://arxiv.org/abs/2410.22920)

C. L. Fefferman, Existence and smoothness of the Navier&ndash;Stokes equation, Clay Mathematics Institute, 2000. [PDF](https://www.claymath.org/wp-content/uploads/2022/06/navierstokes.pdf)

J. Howlett, OpenAI claims blockbuster math breakthrough amid swirl of controversy, _Scientific American_, September 8, 2026. [link](https://www.scientificamerican.com/article/openai-claims-blockbuster-math-breakthrough-amid-swirl-of-controversy/)

K. Kakaes, AI has solved one of math's \$1 million Millennium Prize Problems, _Quanta Magazine_, September 8, 2026. [link](https://www.quantamagazine.org/ai-has-solved-one-of-maths-1-million-millennium-prize-problems-20260908/)

G. Luo, T. Y. Hou, Potentially singular solutions of the 3D axisymmetric Euler equations, _Proc. Natl. Acad. Sci. USA_ **111** (2014) 12968&ndash;12973. [doi:10.1073/pnas.1405238111](https://doi.org/10.1073/pnas.1405238111)

OpenAI, Finite time blowup for Navier&ndash;Stokes, 2026. [PDF](https://cdn.openai.com/pdf/32d9f210-8b73-45e0-91bc-82a30aef8a9a/navier-stokes.pdf); [Lean formalization](https://github.com/openai/NavierStokesAndEuler); [Announcement](https://openai.com/index/navier-stokes-solution/)

R. L. Panton, _Incompressible Flow_, 4th ed., Wiley, 2013. Secs. 11.8 (Oseen vortex), 11.10 (Burgers vortex), 11.12 (von K&aacute;rm&aacute;n viscous pump).